# Indic Med-VQA: layer-wise diagnosis and band-targeted recovery

Runs the whole pipeline on Colab Pro. **Runtime → Change runtime type → L4 (or A100).**
T4/V100 will not work: Gemma 3 needs bf16.

Everything is written to Google Drive (`R` below), and every step resumes, so a disconnected
runtime only loses the item in progress. Run the cells in order; each one says roughly how long it takes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
import os
R = '/content/drive/MyDrive/xling/runs'      # all outputs (data, caches, adapters, reports)
os.environ['XLING_RUNS'] = R
os.makedirs(R, exist_ok=True)
D = f'{R}/data'

## 1. Code and dependencies
Push this repo to GitHub first (private is fine; use a token in the URL), **or** upload the
folder to Drive and change `REPO` to its path.

In [ ]:
REPO = '/content/indic-medvqa-xling'
if not os.path.exists(REPO):
    !git clone https://github.com/sayantansikdar/indic-medvqa-xling.git {REPO}
%cd {REPO}
!git pull -q
!pip install -q -r requirements.txt
!python -m pytest -q tests

## 2. Hugging Face access
Accept the terms on these pages with the same account first:
[MedGemma 1.5 4B](https://huggingface.co/google/medgemma-1.5-4b-it),
[IndicTrans2 en→indic](https://huggingface.co/ai4bharat/indictrans2-en-indic-1B),
[IndicTrans2 indic→en](https://huggingface.co/ai4bharat/indictrans2-indic-en-1B).

In [ ]:
from huggingface_hub import login
login()   # paste a read token

## 3. Data (≈ 30–40 min, mostly the 8 GB ArogyaBodha download)
Keeps items that exist in EN+HI+BN+TA+TE with one image and a consistent answer key,
splits by image hash into test / calib / train (50/20/30).

In [ ]:
!python -m xling.data arogyabodha --out {D}
!python -m xling.data vqarad --out {D}

## 3b. Early smoke test (2 min), before spending time on translation
Uses ArogyaBodha's own Hindi/Tamil text. Check that accuracy is above chance and `letter mass` is high (> 0.5).
Send this output before continuing.

In [ ]:
!python -m xling.infer --items {D}/arogyabodha.jsonl --split test --limit 8 --conds en,hi-mixed,ta-mixed --tag smoke0 --capture

## 4. Translation (≈ 1–2 h on L4, plus ~3 min to build the environment)
`*-mt`: IndicTrans2 from the English source (monolingual; main analysis condition).
`*-tt`: translate-test baseline B1 (Indic → English) for the test split.

IndicTrans2's model code needs transformers < 4.54 and IndicTransToolkit needs transformers < 5, so this
step runs in a fully separate environment (its own Python 3.12, torch and transformers 4.53.3);
everything else stays on transformers 5. Progress is saved after each language, so rerunning resumes.


In [ ]:
%cd /content/indic-medvqa-xling
!pip install -q uv
PY = '/content/it2env/bin/python'
ok = !{PY} -c "import torch, transformers, IndicTransToolkit.processor; print('ENV_OK', torch.__version__, torch.cuda.is_available(), transformers.__version__)" 2>/dev/null
if not any(l.startswith('ENV_OK') for l in ok):
    !rm -rf /content/it2env
    !uv venv --python 3.12 /content/it2env
    !uv pip install --python {PY} torch==2.7.1 -r /content/indic-medvqa-xling/requirements-translate.txt 2>&1 | tail -25
    ok = !{PY} -c "import torch, transformers, IndicTransToolkit.processor; print('ENV_OK', torch.__version__, torch.cuda.is_available(), transformers.__version__)"
print('\n'.join(ok))
assert any(l.startswith('ENV_OK') for l in ok), "translation environment is broken - send the output above"
!{PY} -m xling.translate forward --items {D}/arogyabodha.jsonl
!{PY} -m xling.translate backward --items {D}/arogyabodha.jsonl --from-variant mt --split test
!{PY} -m xling.translate forward --items {D}/vqarad.jsonl


**Only if IndicTrans2 still fails to load or generate** in the cell above: use NLLB-200 instead
(runs in the main environment, not gated, somewhat weaker on Indic languages; note it in the paper).


In [ ]:
# !python -m xling.translate forward --engine nllb --items {D}/arogyabodha.jsonl
# !python -m xling.translate backward --engine nllb --items {D}/arogyabodha.jsonl --from-variant mt --split test
# !python -m xling.translate forward --engine nllb --items {D}/vqarad.jsonl


## 5. Smoke test (2 min) — check accuracy is sane and `letter mass` is high
If `letter mass` is below 0.5 the model is not answering with a letter first; stop and fix the prompt.

In [ ]:
!python -m xling.infer --items {D}/arogyabodha.jsonl --split test --limit 8 --conds en,hi-mt,ta-mt --tag smoke --capture

## 6. Phase A + cache (≈ 2–3 h on L4) — zero-shot baseline B0 and residual/lens/attention cache

In [ ]:
COND = 'en,hi-mt,bn-mt,ta-mt,te-mt,hi-mixed,bn-mixed,ta-mixed,te-mixed'
!python -m xling.infer --items {D}/arogyabodha.jsonl --split calib --conds {COND} --capture
!python -m xling.infer --items {D}/arogyabodha.jsonl --split test  --conds {COND} --capture --fidelity 100
# translate-test (B1) and the stripped-gloss ablation, no cache needed
!python -m xling.infer --items {D}/arogyabodha.jsonl --split test --conds hi-tt,bn-tt,ta-tt,te-tt,hi-pure,bn-pure,ta-pure,te-pure
!python -m xling.infer --items {D}/vqarad.jsonl --split test
!python -m xling.report --root {R}/medgemma/arogyabodha/test --base base
!python -m xling.report --root {R}/medgemma/arogyabodha/test --base base --exclude-sources MedXpertQA_MM_test

## 7. Phase B1–B4 diagnosis (CPU, ≈ 10 min)

In [ ]:
T = f'{R}/medgemma/arogyabodha/test/base'; C = f'{R}/medgemma/arogyabodha/calib/base'
!python -m xling.diagnose --test-dir {T} --calib-dir {C} --variant mt --plot
!python -m xling.diagnose --test-dir {T} --calib-dir {C} --variant mixed --plot
from IPython.display import Image, display
display(Image(f'{T}/diagnosis_mt.png'))

## 8. Phase B5 activation patching (≈ 1–2 h per split on L4)
The band is chosen on **calib** and frozen before anything is evaluated on test.

In [ ]:
MT = 'hi-mt,bn-mt,ta-mt,te-mt'
!python -m xling.patching --items {D}/arogyabodha.jsonl --split calib --conds {MT} --max-items 150 --exclude-sources MedXpertQA_MM_test
!python -m xling.band --patch-dir {R}/medgemma/arogyabodha/calib/patching --plot

In [ ]:
# report-only run on test, including reverse patching (necessity)
!python -m xling.patching --items {D}/arogyabodha.jsonl --split test --conds {MT} --max-items 150 --exclude-sources MedXpertQA_MM_test \
    --modes resid,attn,mlp,win4,win6,win8,ctl-win6,rev-win6
!python -m xling.band --patch-dir {R}/medgemma/arogyabodha/test/patching --plot

## 9. Baseline B3: training-free steering (≈ 40 min)

In [ ]:
!python -m xling.steer --items {D}/arogyabodha.jsonl --variant mt --vectors {C}/steer_mt.npz --layers 6:32:2 --alphas 0.5,1,2 --limit 150
import json
best = json.load(open(f'{C}/steer_sweep_mt.json'))['best']; print(best)
!python -m xling.infer --items {D}/arogyabodha.jsonl --split test --conds en,{MT} --tag steer \
    --steer-file {C}/steer_mt.npz --steer-layer {best['layer']} --steer-alpha {best['alpha']}

## 10. Phase C training (≈ 30–60 min per variant on L4 with `--max-pairs 3000`)
All variants use the same data, steps and (for B4) a matched parameter budget.

In [ ]:
BAND = f'{R}/medgemma/arogyabodha/calib/patching/band.json'
TR = f'{R}/train'
common = f'--items {D}/arogyabodha.jsonl --band-json {BAND} --max-pairs 3000 --seed 0'
!python -m xling.train {common} --placement band   --lam 0.5 --out {TR}/ours
!python -m xling.train {common} --placement band   --lam 0   --out {TR}/band_noalign
!python -m xling.train {common} --placement random --lam 0.5 --out {TR}/random_band
!python -m xling.train {common} --placement all    --lam 0 --match-budget --out {TR}/all_layers

## 11. Phase D evaluation and report

In [ ]:
for tag in ['ours', 'band_noalign', 'random_band', 'all_layers']:
    !python -m xling.infer --items {D}/arogyabodha.jsonl --split test --conds en,{MT},hi-mixed,bn-mixed,ta-mixed,te-mixed --adapter {TR}/{tag} --tag {tag}
    !python -m xling.infer --items {D}/vqarad.jsonl --split test --adapter {TR}/{tag} --tag {tag}
!python -m xling.report --root {R}/medgemma/arogyabodha/test --base base --runs steer,all_layers,random_band,band_noalign,ours
!python -m xling.report --root {R}/medgemma/arogyabodha/test --base base --runs steer,all_layers,random_band,band_noalign,ours --exclude-sources MedXpertQA_MM_test
!python -m xling.report --root {R}/medgemma/vqarad/test --base base --runs all_layers,random_band,band_noalign,ours

## 12. (If compute allows) Qwen2.5-VL-7B control

In [ ]:
!python -m xling.infer --model qwen25vl --items {D}/arogyabodha.jsonl --split calib --conds en,{MT} --capture
!python -m xling.infer --model qwen25vl --items {D}/arogyabodha.jsonl --split test  --conds en,{MT} --capture
!python -m xling.diagnose --test-dir {R}/qwen25vl/arogyabodha/test/base --calib-dir {R}/qwen25vl/arogyabodha/calib/base --variant mt --plot